# Phase 3 — Real Aerial Cropland Images

End-to-end run on 13 manually collected aerial cropland images.

## Central question

> When does MILP outperform greedy? When does greedy win?

On small, simple fields the MILP easily wins — it has time to find a good solution.
On larger, more complex real fields with a crunched solver budget, the MILP can produce
a *worse* makespan than a greedy baseline. This is exactly why the three-tier planner
exists: greedy is the floor, not the enemy.

## What this notebook shows
1. All 13 images → 128×128 priority grids
2. MILP vs greedy sweep across all fields at full budget
3. **Budget sensitivity sweep** — MILP makespan as a function of solver time limit;
   we find the crossover where greedy wins
4. Overlay animation — drone mission on the actual aerial photograph
5. Failure injection + recovery on the most interesting field
6. Monte Carlo worst-case analysis
7. Strip orientation experiment

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

from src.field.ingest import load_image_grid, load_image_directory, load_image_as_array
from src.field.generator import generate_strips
from src.optimizer.milp import DroneSpec, assign_strips
from src.optimizer.planner import plan, PlannerMode, PlannerContext
from src.simulation.engine import simulate
from src.simulation.metrics import (
    compute_metrics, plot_coverage_over_time,
    monte_carlo_analysis, plot_monte_carlo,
)
from src.viz.renderer import animate

%matplotlib inline
print('Imports OK')

## 1. Load all aerial images → 128×128 priority grids

In [ ]:
IMAGE_DIR   = '../aerial_cropland_images/focused'
TARGET_SIZE = 128
CHANNEL     = 'green'
N_DRONES    = 5
DOCK_POS    = [(0, 0)]
STRIP_WIDTH = 2      # 128 rows / sw=2 → 64 strips per field
ORIENTATION = 0.0

print(f'Loading {TARGET_SIZE}×{TARGET_SIZE} grids  channel={CHANNEL}  '
      f'strip_width={STRIP_WIDTH} → 64 strips per field')
print()

all_fields = load_image_directory(IMAGE_DIR, target_size=TARGET_SIZE, channel=CHANNEL)

image_dir_path = Path(IMAGE_DIR)
stem_to_path = {
    f.stem: str(f) for f in image_dir_path.iterdir()
    if f.suffix.lower() in {'.jpg', '.jpeg', '.jfif', '.png'}
}
names = list(all_fields.keys())

## 2. Priority surfaces — all images at 128×128

In [ ]:
n          = len(names)
ncols_plot = 5
nrows_plot = (n + ncols_plot - 1) // ncols_plot

fig, axes = plt.subplots(nrows_plot, ncols_plot,
                         figsize=(ncols_plot * 3.2, nrows_plot * 3.2))
axes = axes.flatten()

for i, name in enumerate(names):
    grid, meta = all_fields[name]
    im = axes[i].imshow(grid, cmap='YlGn', vmin=0, vmax=1, origin='upper')
    axes[i].set_title(f'{name}\nmean={meta["mean_priority"]:.2f}', fontsize=8)
    axes[i].axis('off')
for j in range(n, len(axes)):
    axes[j].axis('off')

fig.colorbar(im, ax=axes[:n].tolist(), shrink=0.5, label='Spray priority')
fig.suptitle(f'Priority grids — {n} aerial images @ {TARGET_SIZE}×{TARGET_SIZE}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../results/phase3_all_priority_grids.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Original vs priority — sanity check

In [ ]:
fig, axes = plt.subplots(n, 2, figsize=(10, n * 2.8))
if n == 1:
    axes = axes[np.newaxis, :]

for i, name in enumerate(names):
    grid, meta = all_fields[name]
    bg = load_image_as_array(stem_to_path[name], target_size=TARGET_SIZE)

    axes[i, 0].imshow(bg)
    axes[i, 0].set_title(f'{name}', fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(grid, cmap='YlGn', vmin=0, vmax=1, origin='upper')
    axes[i, 1].set_title(f'Priority  mean={meta["mean_priority"]:.2f}', fontsize=9)
    axes[i, 1].axis('off')

fig.suptitle(f'Original image vs priority surface  ({TARGET_SIZE}×{TARGET_SIZE})',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/phase3_original_vs_priority.png', dpi=130, bbox_inches='tight')
plt.show()

## 4. MILP vs greedy — sweep across all 13 fields

Full 10-second solver budget. This is the best-case for MILP.  
Fields where `priority_gain` is near zero are where greedy is already competitive.

In [ ]:
rows_summary   = []
field_data     = {}   # name → (strips, h_milp, h_greedy, m_milp, m_greedy)

for name, (grid, meta) in all_fields.items():
    drones = [DroneSpec(id=i) for i in range(N_DRONES)]
    strips = generate_strips(grid, orientation_deg=ORIENTATION, strip_width=STRIP_WIDTH)

    r_milp   = plan(strips, drones, mode=PlannerMode.FULL)
    r_greedy = plan(strips, drones, mode=PlannerMode.HEURISTIC)

    h_milp   = simulate(result=r_milp,   strips=strips, drones=drones,
                        nrows=TARGET_SIZE, ncols=TARGET_SIZE, dock_positions=DOCK_POS)
    h_greedy = simulate(result=r_greedy, strips=strips, drones=drones,
                        nrows=TARGET_SIZE, ncols=TARGET_SIZE, dock_positions=DOCK_POS)

    m_milp   = compute_metrics(h_milp,   strips, TARGET_SIZE, TARGET_SIZE)
    m_greedy = compute_metrics(h_greedy, strips, TARGET_SIZE, TARGET_SIZE)

    field_data[name] = (strips, h_milp, h_greedy, m_milp, m_greedy)

    rows_summary.append({
        'field'          : name,
        'mean_priority'  : round(meta['mean_priority'], 3),
        'milp_makespan'  : m_milp['makespan'],
        'greedy_makespan': m_greedy['makespan'],
        'milp_priority'  : round(m_milp['priority_coverage'], 4),
        'greedy_priority': round(m_greedy['priority_coverage'], 4),
        'priority_gain'  : round(m_milp['priority_coverage'] - m_greedy['priority_coverage'], 4),
        'makespan_ratio' : round(m_milp['makespan'] / m_greedy['makespan'], 4)
                          if m_greedy['makespan'] > 0 else 1.0,
        'milp_solve_s'   : round(r_milp.solve_time, 2),
    })
    tag = '  ← MILP LOSES' if rows_summary[-1]['makespan_ratio'] > 1.0 else ''
    print(f"{name:<42}  ratio={rows_summary[-1]['makespan_ratio']:.3f}  "
          f"Δpriority={rows_summary[-1]['priority_gain']:+.4f}{tag}")

print('\nratio < 1.0 = MILP wins on makespan; ratio > 1.0 = greedy wins')

In [ ]:
import pandas as pd

df = pd.DataFrame(rows_summary).set_index('field')

# Format for display — no jinja2/style dependency
display(df[['mean_priority', 'milp_makespan', 'greedy_makespan', 'makespan_ratio',
            'milp_priority', 'greedy_priority', 'priority_gain', 'milp_solve_s']]
        .round({'mean_priority': 3, 'makespan_ratio': 3,
                'milp_priority': 4, 'greedy_priority': 4,
                'priority_gain': 4, 'milp_solve_s': 2}))

## 4b. GIF animations — all 5 fields (MILP plan)

Overlay: aerial photo as background, coverage tints on top.  
Green = untouched, yellow = in-progress, gray = complete, red = failed/skipped.  
Saved to `results/` for each field.

In [ ]:
import os
os.makedirs('../results', exist_ok=True)

FRAME_SKIP = 8   # ~500 frames at 128×128; adjust up if GIFs are too large

for name in names:
    strips_g, h_milp_g, h_greedy_g, m_milp_g, m_greedy_g = field_data[name]

    bg_arr_g = load_image_as_array(stem_to_path[name], target_size=TARGET_SIZE)
    out_path = f'../results/phase3_{name}_milp.gif'

    animate(
        h_milp_g[::FRAME_SKIP], TARGET_SIZE, TARGET_SIZE,
        interval_ms=120,
        background_image=bg_arr_g,
        save_path=out_path,
        show=False,          # suppress inline display in batch mode
        dock_positions=DOCK_POS,
    )
    n_frames = len(h_milp_g[::FRAME_SKIP])
    size_kb   = os.path.getsize(out_path) // 1024
    print(f'{name:<42}  {n_frames} frames  {size_kb} KB  → {out_path}')

## 5. Budget sensitivity sweep — when does greedy win?

We pick the most complex field (highest priority variance) and increase strip density
to `strip_width=1` → **128 strips, 640 binary variables**. This is the harder version
of the problem: with 64 strips (sw=2) CBC finds a near-optimal solution in under a
second and greedy never wins. At 128 strips the solver needs real time to close the gap.

The greedy baseline is constant — O(n), always instant.  
At short budgets CBC returns a feasible-but-suboptimal assignment that greedy beats.  
As budget grows the MILP converges and eventually wins.

**The crossover point is the minimum viable solver budget — below it, fall back to greedy.
This is exactly what the degraded planner enforces.**

In [ ]:
# Pick the field with highest priority variance — most non-uniform, hardest assignment
variances = {
    name: float(np.array(grid).std())
    for name, (grid, meta) in all_fields.items()
}
sweep_name = max(variances, key=variances.__getitem__)
print(f'Budget sweep field: {sweep_name}  (priority std={variances[sweep_name]:.3f})')

sweep_grid, _ = all_fields[sweep_name]
sweep_drones  = [DroneSpec(id=i) for i in range(N_DRONES)]

# strip_width=1 → 128 strips, 640 binary vars — hard enough that CBC needs real time
SWEEP_STRIP_WIDTH = 1
sweep_strips = generate_strips(sweep_grid, orientation_deg=ORIENTATION, strip_width=SWEEP_STRIP_WIDTH)
print(f'Strips: {len(sweep_strips)}  (strip_width={SWEEP_STRIP_WIDTH})')
print(f'Problem size: {len(sweep_strips)} strips × {N_DRONES} drones = '
      f'{len(sweep_strips) * N_DRONES} binary variables')
print()

# Greedy baseline — instant, deterministic
r_greedy_base   = plan(sweep_strips, sweep_drones, mode=PlannerMode.HEURISTIC)
greedy_makespan = r_greedy_base.makespan
print(f'Greedy baseline: makespan={greedy_makespan:.0f}  solve={r_greedy_base.solve_time:.4f}s')
print()

# Sweep MILP at increasing time budgets
TIME_BUDGETS = [1, 2, 3, 5, 7, 10]
sweep_results = []

for budget in TIME_BUDGETS:
    r     = assign_strips(sweep_strips, sweep_drones, time_limit_seconds=budget)
    ratio = r.makespan / greedy_makespan if greedy_makespan > 0 else 1.0
    sweep_results.append({
        'budget_s'   : budget,
        'makespan'   : r.makespan,
        'solve_s'    : r.solve_time,
        'status'     : r.status,
        'ratio'      : round(ratio, 4),
        'greedy_wins': ratio > 1.0,
    })
    tag = '  ← GREEDY WINS' if ratio > 1.0 else '  (MILP wins)'
    print(f'  budget={budget:2d}s  makespan={r.makespan:.0f}  ratio={ratio:.3f}{tag}')

In [ ]:
budgets   = [r['budget_s'] for r in sweep_results]
makespans = [r['makespan'] for r in sweep_results]
ratios    = [r['ratio']    for r in sweep_results]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# --- Left: absolute makespan vs budget ---
ax1.plot(budgets, makespans, 'o-', color='#1565c0', linewidth=2, markersize=8,
         label='MILP makespan', zorder=3)
ax1.axhline(greedy_makespan, color='#e65100', linewidth=2, linestyle='--',
            label=f'Greedy baseline ({greedy_makespan:.0f}s)')
ax1.fill_between(budgets, makespans, greedy_makespan,
                 where=[m > greedy_makespan for m in makespans],
                 alpha=0.15, color='red', label='Greedy wins zone')
ax1.fill_between(budgets, makespans, greedy_makespan,
                 where=[m <= greedy_makespan for m in makespans],
                 alpha=0.15, color='green', label='MILP wins zone')
ax1.set_xlabel('Solver time budget (seconds)')
ax1.set_ylabel('Makespan (steps)')
ax1.set_title(f'Makespan vs solver budget\n{sweep_name}  {TARGET_SIZE}×{TARGET_SIZE}',
              fontweight='bold')
ax1.legend(fontsize=9)
ax1.grid(True, alpha=0.3)

# --- Right: ratio (MILP / greedy) ---
colours = ['#ef5350' if r > 1.0 else '#66bb6a' for r in ratios]
ax2.bar(budgets, ratios, color=colours, width=0.7, zorder=3)
ax2.axhline(1.0, color='black', linewidth=1.5, linestyle='-')
ax2.set_xlabel('Solver time budget (seconds)')
ax2.set_ylabel('MILP makespan / greedy makespan')
ax2.set_title('Ratio: MILP / greedy\n< 1.0 = MILP wins  |  > 1.0 = greedy wins',
              fontweight='bold')
ax2.text(0.98, 0.96, 'green = MILP wins\nred = greedy wins',
         transform=ax2.transAxes, ha='right', va='top', fontsize=9,
         bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(f'../results/phase3_budget_sensitivity_{sweep_name}.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/phase3_budget_sensitivity_*.png')

**What this shows:**  
At short solver budgets CBC finds a feasible solution quickly but hasn't had time to
close the optimality gap — the assignment it returns can be worse than greedy's
load-balanced round-robin.  
As budget grows the MILP converges and eventually wins.  
The crossover point is the *minimum viable budget* — below it, you should use greedy.

This is exactly what the degraded planner enforces: if `time_budget_seconds` is too short,
fall back to greedy rather than run a half-baked MILP.

## 6. Focus field — largest MILP vs greedy gap at full budget

For deeper analysis: pick the field where MILP earns the most over greedy when given
the full 10-second budget.

In [ ]:
best_row   = max(rows_summary, key=lambda r: abs(r['priority_gain']))
focus_name = best_row['field']
print(f'Focus field: {focus_name}')
print(f'  priority_gain = {best_row["priority_gain"]:+.4f}')
print(f'  makespan_ratio = {best_row["makespan_ratio"]:.3f}  (< 1.0 = MILP faster)')

strips_f, h_milp, h_greedy, m_milp, m_greedy = field_data[focus_name]

fig, ax = plt.subplots(figsize=(10, 4))
plot_coverage_over_time(
    runs={
        f'MILP   (priority={m_milp["priority_coverage"]:.4f},  makespan={m_milp["makespan"]})'  : m_milp['cells_per_step'],
        f'Greedy (priority={m_greedy["priority_coverage"]:.4f}, makespan={m_greedy["makespan"]})': m_greedy['cells_per_step'],
    },
    total_cells=TARGET_SIZE * TARGET_SIZE,
    title=f'Coverage over time — {focus_name}  {TARGET_SIZE}×{TARGET_SIZE}',
    ax=ax,
)
plt.tight_layout()
plt.savefig(f'../results/phase3_coverage_{focus_name}.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Overlay animation — focus field

Aerial photograph as background. Coverage tints overlaid:  
untouched = transparent (photo shows through), yellow = in-progress, gray = complete, red = failed.

In [ ]:
focus_path = stem_to_path[focus_name]
bg_arr     = load_image_as_array(focus_path, target_size=TARGET_SIZE)

FRAME_SKIP = 4   # thin history for GIF (full simulation stored; just skip frames in animation)

anim = animate(
    h_milp[::FRAME_SKIP], TARGET_SIZE, TARGET_SIZE,
    interval_ms=100,
    background_image=bg_arr,
    save_path=f'../results/phase3_{focus_name}_overlay.gif',
    show=True,
    dock_positions=DOCK_POS,
)
print(f'Saved → results/phase3_{focus_name}_overlay.gif  ({len(h_milp[::FRAME_SKIP])} frames)')

## 8. Failure injection — focus field

In [ ]:
drones_f = [DroneSpec(id=i) for i in range(N_DRONES)]
r_f_milp   = plan(strips_f, drones_f, mode=PlannerMode.FULL)
r_f_greedy = plan(strips_f, drones_f, mode=PlannerMode.HEURISTIC)

# Failure timesteps chosen to hit mid-mission on a ~4000-step run
total_steps_est = m_milp['makespan']
fail_t1 = total_steps_est // 4
fail_t2 = total_steps_est // 2

FAILURE_EVENTS = [
    {'timestep': fail_t1, 'drone_id': 0, 'type': 'battery'},
    {'timestep': fail_t2, 'drone_id': 2, 'type': 'mechanical'},
]
print(f'Injecting failures at t={fail_t1} (battery) and t={fail_t2} (mechanical)')

sim_kw = dict(
    strips=strips_f, drones=drones_f,
    nrows=TARGET_SIZE, ncols=TARGET_SIZE,
    dock_positions=DOCK_POS,
    failure_events=FAILURE_EVENTS,
)
h_fail_milp   = simulate(result=r_f_milp,   **sim_kw)
h_fail_greedy = simulate(result=r_f_greedy, **sim_kw)

m_fail_milp   = compute_metrics(h_fail_milp,   strips_f, TARGET_SIZE, TARGET_SIZE)
m_fail_greedy = compute_metrics(h_fail_greedy, strips_f, TARGET_SIZE, TARGET_SIZE)

print(f'MILP replan : coverage={m_fail_milp["coverage_pct"]}%  '
      f'time_to_recovery={m_fail_milp["time_to_recovery"]} steps')
print(f'Greedy      : coverage={m_fail_greedy["coverage_pct"]}%  '
      f'time_to_recovery={m_fail_greedy["time_to_recovery"]} steps')

fig, ax = plt.subplots(figsize=(10, 4))
plot_coverage_over_time(
    runs={
        f'MILP replan   ({m_fail_milp["coverage_pct"]}%)':   m_fail_milp['cells_per_step'],
        f'Greedy replan ({m_fail_greedy["coverage_pct"]}%)': m_fail_greedy['cells_per_step'],
    },
    total_cells=TARGET_SIZE * TARGET_SIZE,
    title=f'Coverage recovery after 2 failures — {focus_name}  {TARGET_SIZE}×{TARGET_SIZE}',
    ax=ax,
)
for ev in FAILURE_EVENTS:
    ax.axvline(ev['timestep'], color='red', linestyle=':', alpha=0.7, linewidth=1.5)
ax.axvline(FAILURE_EVENTS[0]['timestep'], color='red', linestyle=':', alpha=0.7,
           linewidth=1.5, label='Failure event')
ax.legend()
plt.tight_layout()
plt.savefig(f'../results/phase3_failure_{focus_name}.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Failure overlay animation

In [ ]:
anim_fail = animate(
    h_fail_milp[::FRAME_SKIP], TARGET_SIZE, TARGET_SIZE,
    interval_ms=100,
    background_image=bg_arr,
    save_path=f'../results/phase3_{focus_name}_failure_overlay.gif',
    show=True,
    dock_positions=DOCK_POS,
)
print(f'Saved → results/phase3_{focus_name}_failure_overlay.gif')

## 10. Monte Carlo worst-case

In [ ]:
mc = monte_carlo_analysis(
    strips=strips_f, drones=drones_f, result=r_f_milp,
    nrows=TARGET_SIZE, ncols=TARGET_SIZE,
    n_runs=10, failure_prob_per_drone=0.25, seed=42,
)

print(f"Coverage  mean={mc['coverage_pct_mean']}%  "
      f"p5={mc['coverage_pct_p5']}%  p95={mc['coverage_pct_p95']}%")
print(f"Recovery  mean={mc['time_to_recovery_mean']} steps  "
      f"p95={mc['time_to_recovery_p95']} steps")

fig, ax = plt.subplots(figsize=(8, 4))
plot_monte_carlo(mc,
    title=f'Monte Carlo — {focus_name}  {TARGET_SIZE}×{TARGET_SIZE}  (10 runs)',
    ax=ax)
plt.tight_layout()
plt.savefig(f'../results/phase3_mc_{focus_name}.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Strip orientation experiment

Compare 0°, 45°, 90° on images with diagonal or vertical crop rows.

In [ ]:
candidates   = [n for n in names if 'diagonal' in n.lower() or 'vertical' in n.lower()]
orient_field = candidates[0] if candidates else focus_name
print(f'Orientation test: {orient_field}')

og, om = all_fields[orient_field]
drones_o = [DroneSpec(id=i) for i in range(N_DRONES)]

orient_results = {}
for deg in [0, 45, 90]:
    s_o = generate_strips(og, orientation_deg=deg, strip_width=STRIP_WIDTH)
    r_o = plan(s_o, drones_o, mode=PlannerMode.FULL)
    h_o = simulate(result=r_o, strips=s_o, drones=drones_o,
                   nrows=TARGET_SIZE, ncols=TARGET_SIZE, dock_positions=DOCK_POS)
    m_o = compute_metrics(h_o, s_o, TARGET_SIZE, TARGET_SIZE)
    orient_results[deg] = m_o
    print(f"  {deg:3d}°  makespan={m_o['makespan']:5d}  priority={m_o['priority_coverage']:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, deg in enumerate([0, 45, 90]):
    m_o  = orient_results[deg]
    s_o  = generate_strips(og, orientation_deg=deg, strip_width=STRIP_WIDTH)
    sr, sc = zip(*[s.cells[0] for s in s_o])
    axes[i].imshow(og, cmap='YlGn', vmin=0, vmax=1, origin='upper')
    axes[i].scatter(sc, sr, c='red', s=8, zorder=5)
    axes[i].set_title(f'{deg}°\npriority={m_o["priority_coverage"]:.4f}  makespan={m_o["makespan"]}',
                      fontsize=9, fontweight='bold')
    axes[i].axis('off')

fig.suptitle(f'Strip orientation — {orient_field}  {TARGET_SIZE}×{TARGET_SIZE}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(f'../results/phase3_orientation_{orient_field}.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Full comparison table

In [ ]:
display(df[['mean_priority', 'milp_makespan', 'greedy_makespan', 'makespan_ratio',
            'milp_priority', 'greedy_priority', 'priority_gain', 'milp_solve_s']]
        .round({'mean_priority': 3, 'makespan_ratio': 3,
                'milp_priority': 4, 'greedy_priority': 4,
                'priority_gain': 4, 'milp_solve_s': 2}))